In [2]:
###############################################################################
# Asymptotic (Wald) interval for the difference of two correlated ICVs
#   Delta = ICV_X - ICV_Y,  ICV = mean / sd
# Simulation for Models 1-6 (set CASE below)
#
# Data generation: Gaussian copula. For each target Pearson correlation rho,
# the latent correlation rho_L is calibrated once from 1e6 draws and then held
# fixed across replications.
###############################################################################

install.packages(c("foreach", "doParallel", "MASS"))
library(foreach)
library(doParallel)

CASE <- 1          # <-- choose Case 1, 2, 3, 4, 5, or 6
nsim <- 1000       # Monte Carlo replications per scenario
alpha <- 0.05

###############################################################################
# 1. Variance of the ICV difference (delta method)
###############################################################################
var_1f <- function(data){
  1 - mean(data) * (sd(data)^(-4)) * mean((data - mean(data))^3) +
    0.25 * (mean(data)^2) * (sd(data)^(-6)) *
    (mean((data - mean(data))^4) - sd(data)^4)
}

Cov_Tf <- function(data1, data2){
  m1 <- mean(data1); m2 <- mean(data2); s1 <- sd(data1); s2 <- sd(data2)
  c1 <- data1 - m1;  c2 <- data2 - m2
  (1/s2) * (1/s1) * mean(c1 * c2) -
    0.5 * m1 * s1^(-3) * s2^(-1) * mean(c1^2 * c2) -
    0.5 * m2 * s2^(-3) * s1^(-1) * mean(c1 * c2^2) +
    0.25 * m1 * m2 * s1^(-3) * s2^(-3) * (mean(c1^2 * c2^2) - s1^2 * s2^2)
}

# Returns c(lower, upper), or c(NA, NA) if the variance estimate is unusable
asym_ci <- function(x, y, alpha = 0.05){
  if (sd(x) == 0 || sd(y) == 0) return(c(NA, NA))
  n <- length(x)
  d_hat <- mean(x)/sd(x) - mean(y)/sd(y)
  v <- (var_1f(x) + var_1f(y) - 2 * Cov_Tf(x, y)) / n
  if (!is.finite(v) || v <= 0) return(c(NA, NA))
  z <- qnorm(1 - alpha/2)
  c(d_hat - z * sqrt(v), d_hat + z * sqrt(v))
}

###############################################################################
# 2. Data generation (Gaussian copula)
###############################################################################
beta_params_from_mean_cv <- function(m, CV){
  a <- (1 - m)/CV^2 - m
  if (a <= 0) stop("Infeasible Beta parameters.")
  list(alpha = a, beta = a * (1 - m) / m)
}

# Quantile-function factories: the parameters are stored inside each function,
# so the functions work on parallel workers without exporting extra objects.
make_qnorm <- function(mu, s)  { force(mu); force(s); function(u) qnorm(u, mu, s) }
make_qbeta <- function(m, CV)  {
  p <- beta_params_from_mean_cv(m, CV); a <- p$alpha; b <- p$beta
  function(u) qbeta(u, a, b)
}
make_qweibull <- function(k, mean_target) {
  lam <- mean_target / gamma(1 + 1/k); force(k)
  function(u) qweibull(u, shape = k, scale = lam)
}
make_qreflected_weibull <- function(k, mean_target, sd_target) {
  muT <- gamma(1 + 1/k); sdT <- sqrt(gamma(1 + 2/k) - muT^2); force(k)
  function(u) mean_target - sd_target * (qweibull(1 - u, shape = k, scale = 1) - muT) / sdT
}

# Latent correlation giving population Pearson correlation r_target
calib_latent_rho <- function(r_target, qX, qY, N = 1e6, seed = 1){
  if (r_target == 0) return(0)
  set.seed(seed)
  N1 <- rnorm(N); N2 <- rnorm(N)
  X <- qX(pnorm(N1))
  f <- function(rho) cor(X, qY(pnorm(rho * N1 + sqrt(1 - rho^2) * N2))) - r_target
  uniroot(f, c(-0.999, 0.999), tol = 1e-6)$root
}

gen_copula <- function(n, rho_latent, qX, qY, seed){
  set.seed(seed)
  N1 <- rnorm(n); N2 <- rnorm(n)
  list(X = qX(pnorm(N1)),
       Y = qY(pnorm(rho_latent * N1 + sqrt(1 - rho_latent^2) * N2)))
}

###############################################################################
# 3. Model settings (Table 2). All models have equal ICVs, so Delta = 0.
###############################################################################
k_weib <- uniroot(function(k) sqrt(gamma(1 + 2/k)/gamma(1 + 1/k)^2 - 1) - 0.4,
                  c(0.1, 50), tol = 1e-10)$root      # 2.6956

setup <- switch(as.character(CASE),
  "1" = list(qX = make_qnorm(5, 2),   qY = make_qnorm(8, 3.2),
             n = c(5, 10, 30, 100, 500, 1000)),
  "2" = list(qX = make_qnorm(2, 0.6), qY = make_qbeta(0.30, 0.30),
             n = c(30, 100, 500, 1000)),
  "3" = list(qX = make_qnorm(2, 0.6), qY = make_qbeta(0.80, 0.30),
             n = c(30, 100, 500, 1000)),
  "4" = list(qX = make_qbeta(0.70, 0.25), qY = make_qbeta(0.85, 0.25),
             n = c(30, 100, 500, 1000)),
  "5" = list(qX = make_qbeta(0.20, 0.40), qY = make_qbeta(0.35, 0.40),
             n = c(30, 100, 500, 1000)),
  "6" = list(qX = make_qweibull(k_weib, 5),
             qY = make_qreflected_weibull(k_weib, 8, 3.2),
             n = c(30, 100, 500, 1000))
)
qX <- setup$qX
qY <- setup$qY
n_sample_size   <- setup$n
rho_correlation <- c(-0.7, -0.3, 0, 0.3, 0.7)
true_delta      <- 0

###############################################################################
# 4. Parallel backend
###############################################################################
n_cores <- detectCores()
if (n_cores > 1) {
  cl <- makeCluster(n_cores - 1)
  registerDoParallel(cl)
  cat("Registered parallel backend with", n_cores - 1, "cores.\n")
} else {
  registerDoSEQ()
}

###############################################################################
# 5. Simulation
###############################################################################
out <- data.frame()
t0 <- proc.time()

for (rho in rho_correlation) {
  # Model 1 is bivariate normal, so the latent and Pearson correlations coincide
  rho_lat <- if (CASE == 1) rho else calib_latent_rho(rho, qX, qY)
  cat(sprintf("Case %d, rho = %+.1f, latent rho = %+.4f\n", CASE, rho, rho_lat))

  for (n in n_sample_size) {
    results <- foreach(kk = 1:nsim, .combine = rbind) %dopar% {
      set <- gen_copula(n, rho_lat, qX, qY, seed = 2025 + kk)
      ci  <- asym_ci(set$X, set$Y, alpha)
      c(sim_id = kk, Lower = ci[1], Upper = ci[2])
    }
    results <- as.data.frame(results)
    names(results) <- c("sim_id", "Lower", "Upper")

    ok <- is.finite(results$Lower) & is.finite(results$Upper)
    L  <- results$Lower[ok]
    U  <- results$Upper[ok]
    R  <- sum(ok)
    cov_rate <- mean(L <= true_delta & true_delta <= U)

    out <- rbind(out, data.frame(
      Model      = CASE,
      n          = n,
      rho        = rho,
      rho_latent = round(rho_lat, 4),
      R_valid    = R,
      fail       = nsim - R,
      Coverage   = round(cov_rate, 4),
      MCSE       = round(sqrt(cov_rate * (1 - cov_rate) / R), 4),
      Miss_lower = round(mean(true_delta < L), 4),   # true value below the interval
      Miss_upper = round(mean(true_delta > U), 4),   # true value above the interval
      Length     = round(mean(U - L), 4)
    ))
  }
}

if (exists("cl")) stopCluster(cl)
print(proc.time() - t0)

print(out, row.names = FALSE)
write.csv(out, sprintf("asymptotic_case%d.csv", CASE), row.names = FALSE)


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Registered parallel backend with 7 cores.
Case 1, rho = -0.7, latent rho = -0.7000
Case 1, rho = -0.3, latent rho = -0.3000
Case 1, rho = +0.0, latent rho = +0.0000
Case 1, rho = +0.3, latent rho = +0.3000
Case 1, rho = +0.7, latent rho = +0.7000
   user  system elapsed 
  8.539   0.977  11.381 
 Model    n  rho rho_latent R_valid fail Coverage   MCSE Miss_lower Miss_upper
     1    5 -0.7       -0.7     999    1   0.7477 0.0137     0.1171     0.1351
     1   10 -0.7       -0.7    1000    0   0.8700 0.0106     0.0690     0.0610
     1   30 -0.7       -0.7    1000    0   0.9300 0.0081     0.0360     0.0340
     1  100 -0.7       -0.7    1000    0   0.9470 0.0071     0.0230     0.0300
     1  500 -0.7       -0.7    1000    0   0.9460 0.0071     0.0280     0.0260
     1 1000 -0.7       -0.7    1000    0   0.9460 0.0071     0.0280     0.0260
     1    5 -0.3       -0.3     997    3   0.7041 0.0145     0.1484     0.1474
     1   10 -0.3       -0.3    1000    0   0.8370 0.0117     0.0880    